# Experiment 2 — Geometry: Phase 1 (pipeline construction, v2)

Generates its own polygon corpus and the three pipeline categories of
Table 2, under the four state-passing conditions of Table 3. No model calls.

**Output** `geometry_exp2_pipelines.json`, `geometry_exp2_summary.json`, `exp2_runtime.py`

## What v2 fixes

| v1 problem | Effect | v2 |
|---|---|---|
| `get_property` / `get_object` were offered in **every** condition | Under raw, 40–68 % of gated steps had the runtime compute the property; raw became a mixture of two experiments | No read tools in the four primary conditions. Read tool exists only for Experiment 4 |
| Handle-only had a read tool | It was Experiment 4's describe tool, not Table 3's handle-only | Handle-only shows the id and nothing else; its gated score is the chance floor by design |
| Models spent gated steps calling `bounding_box` to see the width | Scored as wrong decisions; below the chance floor | Gated instructions name the admissible calls; anything else is a **deferral**, scored separately |
| Gated pipelines only at depth 1–3 | Compounding not measurable where it matters | Depths 1, 2, 3, 4, 5, 7, 10 for all categories |
| Only the threshold mechanism | Table 2 also specifies property-derived arguments | Both mechanisms |
| Handle-only had `get_property` | It exposed the property, i.e. it was handle+summary on demand | Handle conditions get `get_object(id)` only: the WKT, never a property |
| Decision margins 2–10 % only | Every model at the chance floor; no room for tier or capacity effects | Two pre-registered bands: tight 2–10 %, wide 25–50 % |
| Prediction cell calibrated on observed Exp2 numbers | Not a prediction | Predictions use Exp1 accuracies and chance floors only |
| Multi-object had only the selection mechanism while property-conditioned had threshold, boolean and derived | The two category means averaged over different mechanism and band mixes and were not comparable (v7 inversion) | 2 x 4 factorial: both categories carry threshold, boolean and derived on the same gates and bands; selection is multi-object's own fourth mechanism |
| Width and vertex count pooled into the headline as gates | They are *local* properties (95–100 % in Exp1); they hid the bottleneck in the V4-Flash pilot (raw 0.77 pooled, 0.40 on the area gate) | Gates carry a `gate_locality`; headline Table 4 = global gates (area, perimeter, convexity); local gates are reported as the control |
| 24 pass-through pipelines per depth | Depth curves too noisy for an EAF | Cell sizes are a parameter; default 36–40 per (category, depth) |
| `buffer` used round joins | Vertex count exploded along a pipeline | Mitre joins; tier is preserved along the pipeline |
| 2-decimal floats | Not the protocol's format; Exp1 v6 showed decimals push global properties to the floor | Integer coordinates in every tier and for every intermediate object |

## Both orderings of Table 4 are predictions, not targets

Down the columns: pass-through ≥ property-conditioned ≥ multi-object, because a
two-way threshold has a 50 % floor and a three-way selection a 33 % floor.
Across the rows: raw ≤ augmented ≈ handle+summary. Handle-only sits at the
chance floor on gated steps, so whether raw ≥ handle-only depends on the
margin band. That is recorded as a prediction and reported either way.

## [CELL 1] Environment

In [ ]:
!pip install -q shapely scipy

## [CELL 2] Runtime

This cell writes `exp2_runtime.py`. The Phase 2 notebook writes the identical
file, so construction and execution share one definition of every operation,
condition and scoring rule.

In [ ]:
%%writefile exp2_runtime.py
"""
Experiment 2 - Geometry runtime (v2).  Written identically by the Phase 1 and
Phase 2 notebooks so pipeline construction and pipeline execution share one
definition of every operation, condition and scoring rule.

Changes from v1 (see PHASE1_README):
  * No auxiliary read tools in the four primary conditions.  Table 3 defines
    handle-only as "object id; no property information"; raw and augmented
    must force the model to reconstruct properties from text.  A `describe`
    read tool exists only for Experiment 4 and is switched on explicitly.
  * Gated steps name the admissible tools.  A call to any other tool is a
    DEFERRAL (the model tried to make the runtime compute the property) and is
    scored separately from a wrong decision.
  * buffer uses mitre joins so the vertex count, and therefore the tier, is
    preserved along a pipeline.
"""
import math
import random
import re

from shapely import affinity, delaunay_triangles, voronoi_polygons
from shapely import wkt as shapely_wkt
from shapely.geometry import MultiPoint, Polygon
from shapely.geometry.polygon import orient
from shapely.ops import nearest_points as _nearest_points

WKT_PRECISION = 0            # integer grid: every object, initial or intermediate, is serialized with integer coordinates

# ===========================================================================
# Polygon generator (identical to Experiment 1 Phase 1 v3)
# ===========================================================================


# A tier is (vmin, vmax, coord_lo, coord_hi, coord_type).  coord_type "int"
# rounds to integers, "float2" to two decimals.
TIERS = {"simple": (3, 8, 0.0, 1000.0, "int"),      # protocol vertex ranges, integer grid in every tier
         "medium": (10, 20, 0.0, 1000.0, "int"),
         "hard":   (20, 40, 0.0, 1000.0, "int")}
PRECISION = WKT_PRECISION
# Target area is a FRACTION of the coordinate span squared, log-uniform, so it
# is matched across tiers whether or not the tiers share a coordinate range.
AREA_FRAC_LO, AREA_FRAC_HI = 0.004, 0.12
# Aspect-ratio targets (bbox width / height), log-uniform.  Irregular polygons
# are elongated (stretch 2-5 on one axis, either axis) as in the protocol.
ASPECT_RANGE = {"convex": (0.5, 2.0), "concave": (0.5, 2.0), "irregular": (2.0, 5.0)}
MIN_FILL = 0.05                              # area / bbox area
MIN_HULL_DEFICIT = 0.03                      # non-convex polygons must miss >= 3% of their hull area


# ---------------------------------------------------------------------------
# Unit-scale shape generators.  All take (rng, n, bias) and return a Polygon
# centred near the origin.  `bias` in [0, 0.5] pushes the centroid away from
# the bbox centre in a random direction (limacon r = 1 + bias*cos(theta)).
# ---------------------------------------------------------------------------

def _angles(rng, n, alpha=0.7):
    """n angles around the circle.  Gaps are a mix of a uniform share and a
    random exponential share, so the minimum gap is at least (1-alpha)*2pi/n
    (no near-coincident vertices) while edge lengths still vary a lot."""
    e = [rng.expovariate(1.0) for _ in range(n)]
    tot = sum(e)
    gaps = [2 * math.pi * ((1 - alpha) / n + alpha * ei / tot) for ei in e]
    start = rng.uniform(0, 2 * math.pi)
    ang, t = [], start
    for g in gaps[:-1]:
        t += g
        ang.append(t % (2 * math.pi))
    ang.append(start)
    return sorted(ang)


def _limacon(theta, bias, phi):
    return 1.0 + bias * math.cos(theta - phi)


def unit_convex(rng, n, bias):
    """Vertices on the convex limacon r = 1 + b cos(t), b <= 0.5, in angular
    order: exact vertex count, guaranteed convex."""
    ang = _angles(rng, n)
    if ang is None:
        return None
    phi = rng.uniform(0, 2 * math.pi)
    return Polygon([(_limacon(t, bias, phi) * math.cos(t),
                     _limacon(t, bias, phi) * math.sin(t)) for t in ang])


def unit_concave(rng, n, bias):
    """Star-shaped by angular sweep with random radii; mild dents."""
    ang = _angles(rng, n)
    if ang is None:
        return None
    phi = rng.uniform(0, 2 * math.pi)
    rmin = rng.uniform(0.40, 0.75)
    return Polygon([(_limacon(t, bias, phi) * rng.uniform(rmin, 1.0) * math.cos(t),
                     _limacon(t, bias, phi) * rng.uniform(rmin, 1.0) * math.sin(t)) for t in ang])


def unit_irregular(rng, n, bias):
    """Protocol class 'irregular/elongated': radial generation with non-uniform
    vertex spacing and radii, followed (in place()) by an affine stretch of
    2-5 on one axis.  No spikes: every vertex sits on the same body, so the
    bbox diagonal is set by the body and not by a few outliers."""
    ang = _angles(rng, n, alpha=0.9)          # strongly non-uniform spacing
    phi = rng.uniform(0, 2 * math.pi)
    rmin = rng.uniform(0.45, 0.85)
    pts = []
    for t in ang:
        r = rng.uniform(rmin, 1.0) * _limacon(t, bias, phi)
        pts.append((r * math.cos(t), r * math.sin(t)))
    return Polygon(pts)


def unit_convex_sharp(rng, n, bias):
    """Points spread over the rounded corners of a random triangle or
    quadrilateral: exactly n vertices, convex, and as 'pointed' as `bias`
    makes it (bias 0 = nearly round, bias 0.5 = nearly the base polygon).
    Round shapes with many vertices have their centroid glued to the bbox
    centre; this generator is what lets a 40-vertex convex polygon have a
    centroid well away from it."""
    m = rng.choice([3, 3, 4])
    base = None
    for _ in range(30):
        pts = [(math.cos(t), math.sin(t)) for t in _angles(rng, m, alpha=0.6)]
        b = Polygon(pts)
        if b.is_valid and b.area > 0.4:
            base = b
            break
    if base is None:
        return None
    r = 0.95 - 1.4 * bias                        # corner radius 0.25..0.95 (tighter collapses on an integer grid)
    ring = list(base.exterior.coords)[:-1]
    arcs = []
    for i in range(m):
        p0, p1, p2 = ring[i - 1], ring[i], ring[(i + 1) % m]
        a0 = math.atan2(p1[1] - p0[1], p1[0] - p0[0]) - math.pi / 2   # outward normal of edge in
        a1 = math.atan2(p2[1] - p1[1], p2[0] - p1[0]) - math.pi / 2   # outward normal of edge out
        turn = (a1 - a0) % (2 * math.pi)
        arcs.append((p1, a0, turn))
    total = sum(a[2] for a in arcs)
    pts = []
    # allocate vertices to arcs in proportion to turn angle, at least one each
    alloc = [1] * m
    for _ in range(n - m):
        u = rng.uniform(0, total); acc = 0
        for i, a in enumerate(arcs):
            acc += a[2]
            if u <= acc:
                alloc[i] += 1
                break
        else:
            alloc[-1] += 1
    for (c, a0, turn), k in zip(arcs, alloc):
        # evenly spaced along the arc with jitter, so no two vertices collapse on the grid
        for i in range(k):
            t = (i + 0.5 + rng.uniform(-0.3, 0.3)) / k
            ang = a0 + t * turn
            pts.append((c[0] + r * math.cos(ang), c[1] + r * math.sin(ang)))
    hull = Polygon(pts).convex_hull
    if hull.geom_type != "Polygon" or len(hull.exterior.coords) - 1 != n:
        return None
    return hull


def projective_squeeze(poly, k, phi):
    """(x, y) -> (x, y) / (1 + k x) after rotating by -phi.  Projective maps
    preserve convexity and simplicity; this one turns a round shape into an
    egg and moves the centroid away from the bbox centre by an amount that
    grows with k.  The polygon is first normalised so |coord| <= 1."""
    ring = list(poly.exterior.coords)[:-1]
    R = max(math.hypot(x, y) for x, y in ring)
    c, s = math.cos(phi), math.sin(phi)
    out = []
    for x, y in ring:
        x, y = x / R, y / R
        u, v = c * x + s * y, -s * x + c * y
        d = 1 + k * u
        u, v = u / d, v / d
        out.append((c * u - s * v, s * u + c * v))
    return Polygon(out)


GENERATORS = {"convex": unit_convex, "concave": unit_concave, "irregular": unit_irregular}


# ---------------------------------------------------------------------------
# Placement: stretch to target aspect, rescale to target area, translate into
# bounds, round to PRECISION.  The rounded polygon is what gets validated.
# ---------------------------------------------------------------------------

def _round(v, ctype):
    return float(round(v)) if ctype == "int" else round(v, PRECISION)


def place(poly, target_area, target_aspect, rng, tier="hard"):
    vmin, vmax, lo, hi, ctype = TIERS[tier]
    if poly is None or poly.is_empty or poly.area <= 0:
        return None
    minx, miny, maxx, maxy = poly.bounds
    w, h = maxx - minx, maxy - miny
    if w <= 0 or h <= 0:
        return None
    poly = affinity.scale(poly, xfact=target_aspect / (w / h), yfact=1.0, origin="centroid")
    s = math.sqrt(target_area / poly.area)
    poly = affinity.scale(poly, xfact=s, yfact=s, origin="centroid")
    minx, miny, maxx, maxy = poly.bounds
    if (maxx - minx) >= (hi - lo) or (maxy - miny) >= (hi - lo):
        return None
    poly = affinity.translate(poly, xoff=rng.uniform(lo - minx, hi - maxx),
                                    yoff=rng.uniform(lo - miny, hi - maxy))
    return Polygon([(_round(x, ctype), _round(y, ctype)) for x, y in list(poly.exterior.coords)[:-1]])


# ---------------------------------------------------------------------------
# Validity gate.  Every rejection has a named reason.
# ---------------------------------------------------------------------------

def is_convex(poly, tol=1e-9):
    a = poly.area
    return a > 0 and abs(poly.convex_hull.area - a) <= tol * max(a, 1.0)


def check_validity(poly, tier, expected_convex):
    vmin, vmax, lo, hi, ctype = TIERS[tier]
    if poly is None:
        return False, "generation_failed"
    if not poly.is_valid:
        return False, "not_valid"
    if not poly.is_simple:
        return False, "not_simple"
    ring = list(poly.exterior.coords)[:-1]
    n = len(ring)
    if not (vmin <= n <= vmax):
        return False, "vertex_count_out_of_range"
    for i in range(n):
        if ring[i] == ring[(i + 1) % n]:
            return False, "duplicate_adjacent"
    crosses = []
    for i in range(n):
        (x0, y0), (x1, y1), (x2, y2) = ring[i - 1], ring[i], ring[(i + 1) % n]
        crosses.append((x1 - x0) * (y2 - y1) - (y1 - y0) * (x2 - x1))
    if min(abs(c) for c in crosses) < 1e-3:
        return False, "collinear"
    sign_convex = all(c > 0 for c in crosses) or all(c < 0 for c in crosses)
    if poly.area <= 10:
        return False, "area_too_small"
    minx, miny, maxx, maxy = poly.bounds
    if min(minx, miny) < lo or max(maxx, maxy) > hi:
        return False, "out_of_bounds"
    fr = poly.area / ((maxx - minx) * (maxy - miny))
    if fr < MIN_FILL:
        return False, "sliver_low_fill"
    if fr >= FILL_MAX:
        return False, "too_round_high_fill"
    if is_convex(poly) != expected_convex or sign_convex != expected_convex:
        return False, "convexity_mismatch"
    if not expected_convex and 1 - poly.area / poly.convex_hull.area < MIN_HULL_DEFICIT:
        return False, "dent_too_shallow"
    return True, None


# ---------------------------------------------------------------------------
# Nuisance quantities used for cross-tier matching.
# ---------------------------------------------------------------------------

def bbox_diagonal(poly):
    minx, miny, maxx, maxy = poly.bounds
    return math.hypot(maxx - minx, maxy - miny)


def centroid_offset_norm(poly):
    """Distance from the true centroid to the bbox centre, / bbox diagonal.
    This is the quantity that decides whether 'answer the middle of the box'
    passes a centroid question at a given tolerance."""
    minx, miny, maxx, maxy = poly.bounds
    c = poly.centroid
    return math.hypot(c.x - (minx + maxx) / 2, c.y - (miny + maxy) / 2) / bbox_diagonal(poly)


def fill_ratio(poly):
    minx, miny, maxx, maxy = poly.bounds
    return poly.area / ((maxx - minx) * (maxy - miny))


# Guessability bands.  Each (tier, shape) cell must contain the same number of
# polygons in each band, so no tier is more guessable than another.
#   offset band: distance centroid -> bbox centre, / bbox diagonal (centroid guessability)
#   fill band:   area / bbox area (area and perimeter guessability from the bbox)
OFFSET_BANDS = [(0.00, 0.02), (0.02, 0.04), (0.04, 0.07), (0.07, 10.0)]
FILL_BANDS = [(0.0, 0.52), (0.52, 0.62), (0.62, 0.70), (0.70, 0.76)]
FILL_MAX = 0.76                              # near-ellipses (fill -> pi/4) are excluded in every tier


def fill_band(fr):
    for i, (lo, hi) in enumerate(FILL_BANDS):
        if lo <= fr < hi:
            return i
    return len(FILL_BANDS) - 1


# Joint (offset band, fill band) cells and their share of each shape cell.
# Only cells reachable in EVERY tier are used (measured on the integer grid:
# e.g. a 40-vertex convex polygon cannot have both a far-off centroid and a
# low fill).  The same shares apply in all three tiers, so no tier is more
# guessable than another on either quantity.
JOINT_SHARES = {
    # convex: a 20-40 vertex convex polygon on the integer grid cannot have fill < 0.52,
    # so convex cells never use fill band 0 in any tier (triangles appear in the irregular class).
    "convex":    {(0, 1): 5, (0, 2): 5, (0, 3): 4, (1, 1): 6, (1, 2): 8, (1, 3): 4, (2, 1): 6, (2, 2): 7, (2, 3): 3, (3, 2): 2},
    "concave":   {(0, 1): 2, (0, 2): 2, (1, 0): 2, (1, 1): 4, (1, 2): 3, (1, 3): 1, (2, 0): 2, (2, 1): 4, (2, 2): 2, (3, 0): 3},
    "irregular": {(0, 1): 2, (0, 2): 2, (1, 0): 2, (1, 1): 4, (1, 2): 3, (1, 3): 1, (2, 0): 2, (2, 1): 4, (2, 2): 2, (3, 0): 3},
}


def joint_quota(shape, n):
    """Scale JOINT_SHARES[shape] to n slots (largest-remainder rounding)."""
    sh = JOINT_SHARES[shape]; tot = sum(sh.values())
    raw = {k: v * n / tot for k, v in sh.items()}
    q = {k: int(v) for k, v in raw.items()}
    for k, _ in sorted(raw.items(), key=lambda kv: -(kv[1] - int(kv[1])))[: n - sum(q.values())]:
        q[k] += 1
    return q


def joint_band(poly):
    return (offset_band(centroid_offset_norm(poly)), fill_band(fill_ratio(poly)))


def offset_band(d):
    for i, (lo, hi) in enumerate(OFFSET_BANDS):
        if lo <= d < hi:
            return i
    return len(OFFSET_BANDS) - 1


def span(tier):
    return TIERS[tier][3] - TIERS[tier][2]


def draw_targets(rng, shape):
    """(area fraction of span^2, aspect ratio) from the shared distributions."""
    af = math.exp(rng.uniform(math.log(AREA_FRAC_LO), math.log(AREA_FRAC_HI)))
    lo, hi = ASPECT_RANGE[shape]
    ar = math.exp(rng.uniform(math.log(lo), math.log(hi)))
    if shape == "irregular" and rng.random() < 0.5:
        ar = 1.0 / ar                      # stretch along either axis
    return af, ar


def make_one(rng, shape, tier, target_area, target_aspect, tries=300):
    """Rejection-sample one valid polygon of the given shape class and tier.
    The two shape knobs (limacon bias, projective squeeze) are drawn at random
    on every attempt from ranges shared across tiers.  Returns (poly, knobs,
    rejection_reasons)."""
    vmin, vmax = TIERS[tier][:2]
    reasons = {}
    for _ in range(tries):
        n = rng.randint(vmin, vmax)
        bias = rng.uniform(0.0, 0.5)
        squeeze = rng.uniform(0.0, 0.85)
        if shape == "convex" and rng.random() < 0.6:
            unit = unit_convex_sharp(rng, n, bias); gen = "convex_sharp"
        else:
            unit = GENERATORS[shape](rng, n, bias); gen = shape
        if unit is None:
            reasons["generation_failed"] = reasons.get("generation_failed", 0) + 1
            continue
        unit = projective_squeeze(unit, squeeze, rng.uniform(0, 2 * math.pi))
        p = place(unit, target_area, target_aspect, rng, tier)
        ok, why = check_validity(p, tier, shape == "convex")
        if ok:
            p = orient(p, 1.0)   # canonical ccw; Phase 1 flips exactly half
            return p, {"generator": gen, "bias": round(bias, 4), "squeeze": round(squeeze, 4)}, reasons
        reasons[why] = reasons.get(why, 0) + 1
    return None, None, reasons


# ===========================================================================
# The twelve specification operations
# ===========================================================================

def op_convex_hull(p):        return p.convex_hull
def op_buffer(p, distance):   return p.buffer(distance, join_style="mitre", mitre_limit=5.0)
def op_simplify(p, tol):      return p.simplify(tol, preserve_topology=True)
def op_rotate(p, angle):      return affinity.rotate(p, angle, origin="centroid")
def op_translate(p, dx, dy):  return affinity.translate(p, xoff=dx, yoff=dy)
def op_scale(p, factor):      return affinity.scale(p, xfact=factor, yfact=factor, origin="centroid")
def op_intersection(a, b):    return a.intersection(b)
def op_union(a, b):           return a.union(b)
def op_centroid(p):           return p.centroid
def op_triangulate(p):        return delaunay_triangles(MultiPoint(_verts(p)))
def op_voronoi(p):            return voronoi_polygons(MultiPoint(_verts(p)))
def op_nearest_points(a, b):  return _nearest_points(a, b)

def op_bounding_box(p):
    a, b, c, d = p.bounds
    return Polygon([(a, b), (c, b), (c, d), (a, d)])

def _verts(p):
    return list(p.exterior.coords)[:-1]


CHAIN_UNARY, CHAIN_BINARY, TERMINAL = "chain_unary", "chain_binary", "terminal"

OPS = {
    "convex_hull":    (op_convex_hull,    CHAIN_UNARY,  1, []),
    "buffer":         (op_buffer,         CHAIN_UNARY,  1, ["distance"]),
    "simplify":       (op_simplify,       CHAIN_UNARY,  1, ["tolerance"]),
    "bounding_box":   (op_bounding_box,   CHAIN_UNARY,  1, []),
    "rotate":         (op_rotate,         CHAIN_UNARY,  1, ["angle"]),
    "translate":      (op_translate,      CHAIN_UNARY,  1, ["dx", "dy"]),
    "scale":          (op_scale,          CHAIN_UNARY,  1, ["factor"]),
    "intersection":   (op_intersection,   CHAIN_BINARY, 2, []),
    "union":          (op_union,          CHAIN_BINARY, 2, []),
    "centroid":       (op_centroid,       TERMINAL,     1, []),
    "triangulate":    (op_triangulate,    TERMINAL,     1, []),
    "voronoi":        (op_voronoi,        TERMINAL,     1, []),
    "nearest_points": (op_nearest_points, TERMINAL,     2, []),
}
SPEC_OPS      = list(OPS)
CHAINABLE_OPS = [k for k, v in OPS.items() if v[1] in (CHAIN_UNARY, CHAIN_BINARY)]
TERMINAL_OPS  = [k for k, v in OPS.items() if v[1] == TERMINAL]
DESCRIBE_TOOL = "describe"          # Experiment 4 only

def n_operands(t):  return OPS[t][2]
def arg_names(t):   return OPS[t][3]
def is_terminal(t): return OPS[t][1] == TERMINAL


def to_wkt(poly, precision=WKT_PRECISION):
    if precision == 0:
        body = ", ".join(f"{int(round(x))} {int(round(y))}" for x, y in poly.exterior.coords)
    else:
        body = ", ".join(f"{x:.{precision}f} {y:.{precision}f}" for x, y in poly.exterior.coords)
    return f"POLYGON(({body}))"


def snap(poly):
    """Round to serialization precision so the stored form is the computed form."""
    return shapely_wkt.loads(to_wkt(poly))


def is_runtime_valid(poly):
    try:
        return (poly is not None and not poly.is_empty and poly.geom_type == "Polygon"
                and poly.is_valid and poly.is_simple and poly.area > 0)
    except Exception:
        return False


def apply_op(tool, operands, args, strict=False):
    """strict=True (pipeline construction): a non-Polygon result is an error
    rather than being replaced by its convex hull."""
    fn, klass, n_ops, names = OPS[tool]
    if len(operands) != n_ops:
        raise ValueError(f"{tool} takes exactly {n_ops} object argument(s), got {len(operands)}")
    if len(args) != len(names):
        raise ValueError(f"{tool} takes exactly {len(names)} numeric argument(s), got {len(args)}")
    out = fn(*operands, *args)
    if klass == TERMINAL:
        return out
    if out.geom_type != "Polygon":
        if strict:
            raise ValueError(f"{tool} produced {out.geom_type}")
        out = out.convex_hull
    return snap(orient(out, 1.0))


# ===========================================================================
# Properties and the deterministic summary (Table 3)
# ===========================================================================

def properties(poly):
    minx, miny, maxx, maxy = poly.bounds
    ring = list(poly.exterior.coords)
    edges = [math.dist(ring[i], ring[i + 1]) for i in range(len(ring) - 1)]
    mean_e = sum(edges) / len(edges)
    w, h = maxx - minx, maxy - miny
    return {
        "vertex_count": len(ring) - 1,
        "area": round(poly.area, 4),
        "perimeter": round(poly.length, 4),
        "width": round(w, 4),
        "height": round(h, 4),
        "aspect_ratio": round(w / h, 4),
        "edge_length_variance": round(sum((e - mean_e) ** 2 for e in edges) / len(edges), 4),
        "bbox": [round(minx, 4), round(miny, 4), round(maxx, 4), round(maxy, 4)],
        "centroid": [round(poly.centroid.x, 4), round(poly.centroid.y, 4)],
        "centroid_y": round(poly.centroid.y, 4),
        "convex": bool(is_convex(poly)),
        "is_simple": bool(poly.is_simple),
    }


GATE_VALUE = {"area": lambda q: q["area"], "perimeter": lambda q: q["perimeter"],
              "aspect_ratio": lambda q: q["aspect_ratio"],
              "edge_length_variance": lambda q: q["edge_length_variance"],
              "centroid_y": lambda q: q["centroid_y"],
              "width": lambda q: q["width"], "vertex_count": lambda q: q["vertex_count"]}
GATE_PHRASE = {"area": "area", "perimeter": "perimeter",
               "aspect_ratio": "aspect ratio (bounding-box width divided by bounding-box height)",
               "edge_length_variance": "variance of the edge lengths",
               "centroid_y": "northernmost centroid (largest centroid y coordinate)",
               "width": "horizontal extent (bounding-box width)", "vertex_count": "number of vertices"}


def summary_line(poly):
    """The deterministic property summary of Table 3.  It states PRIMITIVE
    measurements.  Composite quantities the pipeline may ask about (the aspect
    ratio = width / height) are deliberately not pre-computed: an augmented or
    handle+summary step still has to combine two stated numbers.  That is what a
    real framework summary looks like, and it means those conditions measure
    read-and-use rather than read-and-copy."""
    q = properties(poly)
    return ("[Properties]\n"
            f"  area = {q['area']}\n"
            f"  perimeter = {q['perimeter']}\n"
            f"  width = {q['width']}\n"
            f"  height = {q['height']}\n"
            f"  edge_length_variance = {q['edge_length_variance']}\n"
            f"  vertex_count = {q['vertex_count']}\n"
            f"  centroid = ({q['centroid'][0]}, {q['centroid'][1]})\n"
            f"  bbox = ({q['bbox'][0]}, {q['bbox'][1]}, {q['bbox'][2]}, {q['bbox'][3]})\n"
            f"  convex = {q['convex']}\n"
            f"  is_simple = {q['is_simple']}")


# ===========================================================================
# The four state-passing conditions (Table 3) and the two propagation modes
# ===========================================================================

CONDITIONS = ["raw", "augmented", "handle", "handle_sum"]
MODES = ["cascading", "oracle"]
TEXT_CONDITIONS = {"raw", "augmented"}
HANDLE_CONDITIONS = {"handle", "handle_sum"}


def operand_labels(visible_handles, condition):
    if condition in HANDLE_CONDITIONS:
        return {h: h for h in visible_handles}
    return {h: f"Polygon {chr(65 + i)}" for i, h in enumerate(visible_handles)}


def render_state(visible, condition):
    labels = operand_labels([h for h, _ in visible], condition)
    parts = []
    for h, p in visible:
        lab = labels[h]
        if condition == "raw":
            parts.append(f"{lab}:\n{to_wkt(p)}")
        elif condition == "augmented":
            parts.append(f"{lab}:\n{to_wkt(p)}\n{summary_line(p)}")
        elif condition == "handle":
            parts.append(lab)
        elif condition == "handle_sum":
            parts.append(f"{lab}\n{summary_line(p)}")
        else:
            raise ValueError(condition)
    return "\n\n".join(parts)


def render_instruction(template, visible_handles, condition):
    labels = operand_labels(visible_handles, condition)
    return template.format(**{f"o{i}": labels[h] for i, h in enumerate(visible_handles)})


def object_matches(ref, expected_poly, expected_handle, condition):
    """handle conditions: the id must match.  text conditions: the WKT must
    parse and agree with the expected geometry to 1% (area and bounds)."""
    if ref is None:
        return False, "missing"
    s = str(ref).strip()
    if condition in HANDLE_CONDITIONS:
        return (s == expected_handle), ("ok" if s == expected_handle else "wrong_handle")
    m = re.search(r"POLYGON\s*\(\(.*?\)\)", s, re.S | re.I)
    if not m:
        return False, "not_wkt"
    try:
        got = shapely_wkt.loads(m.group(0))
    except Exception:
        return False, "unparseable_wkt"
    if got.is_empty or not got.is_valid:
        return False, "invalid_wkt"
    if abs(got.area - expected_poly.area) / max(abs(expected_poly.area), 1e-9) > 0.01:
        return False, "wrong_geometry"
    b1, b2 = got.bounds, expected_poly.bounds
    span = max(b2[2] - b2[0], b2[3] - b2[1], 1e-9)
    if max(abs(x - y) for x, y in zip(b1, b2)) / span > 0.01:
        return False, "wrong_geometry"
    return True, "ok"

In [ ]:
import json, math, random, re
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
from scipy.stats import ks_2samp, norm
import shapely
from exp2_runtime import *      # noqa

print("shapely", shapely.__version__)
BASE = Path.cwd()
OUT_PIPELINES = BASE / "geometry_exp2_pipelines.json"
OUT_SUMMARY   = BASE / "geometry_exp2_summary.json"

SEED = 20260914
N_POLYGONS_PER_TIER = 120                       # 60 convex / 30 concave / 30 irregular

# ---- design constants (pre-registered) -----------------------------------
DEPTHS = [1, 2, 3, 4, 5, 7, 10]                 # specification 5.3, all three categories
PER_CELL = {"pass_through": 24, "property_conditioned": 45, "multi_object": 63}   # PT: mult of 3; PC 15 combos x3; MO 21 combos x3
MARGIN_BANDS = {"tight": (0.02, 0.10), "wide": (0.25, 0.50)}
VC_BANDS     = {"tight": (0.0001, 0.15), "wide": (0.25, 0.60)}   # vertex_count is discrete
# Gate locality (Exp1 taxonomy).  Headline Table 4 uses GLOBAL gates only; local
# gates are the control: with a local gate the raw pipeline should be near
# ceiling, which shows the bottleneck is specific to derived properties.
GATE_LOCALITY = {"area": "global", "perimeter": "global", "convex": "global",
                 "aspect_ratio": "global", "edge_length_variance": "global", "centroid_y": "global",
                 "width": "local", "vertex_count": "local"}
PC_BOOLEAN_GATES = ["convex"]                  # Table 2's own example: "If the polygon is convex, ..."
# Table 2 lists two property-conditioned mechanisms: a branching condition and a
# property-derived argument.  Derived templates: (gate, tool, fraction, phrase)
# Signs alternate along the pipeline so the object stays bounded over 10 steps
# (buffering by +10% of the perimeter ten times would grow it 130x).
PC_DERIVED = [("perimeter", "buffer", (0.05, -0.04), "5% of the perimeter", "minus 4% of the perimeter"),
              ("aspect_ratio", "rotate", (30.0, 25.0),
               "30 times the aspect ratio (bounding-box width divided by bounding-box height)",
               "25 times the aspect ratio (bounding-box width divided by bounding-box height)"),
              ("width", "translate", (0.50, -0.50), "50% of the horizontal extent (bounding-box width)",
               "minus 50% of the horizontal extent (bounding-box width)")]
ARG_TOL_DERIVED = 0.05                          # a value the model had to COMPUTE
MO_GATES = ["area", "perimeter", "aspect_ratio", "centroid_y", "width", "vertex_count"]   # last two = local controls
MO_N_OBJECTS = 4                                # selection chance floor 1/4, vs 1/2 for a two-way threshold
PC_SCALE = 0.85                                 # shrink branch; the buffer branch grows ~10%, so balanced branches stay in range on the integer grid
PC_TOOLS = {"above": ("scale", [PC_SCALE]), "below": ("buffer", None)}   # buffer distance = 5% of diagonal
TIER_ORDER = ["simple", "medium", "hard"]

## [CELL 3] Corpus

Same generator as Experiment 1 v3: protocol vertex ranges (3–8 / 10–20 /
20–40), integer coordinates in [0, 1000] in every tier, area and aspect
targets drawn from one shared distribution. Intermediate objects produced
by the runtime are snapped to the same integer grid, so the serialization
format never changes along a pipeline.

In [ ]:
def build_corpus(rng):
    corpus = []
    plan = (["convex"] * (N_POLYGONS_PER_TIER // 2) + ["concave"] * (N_POLYGONS_PER_TIER // 4)
            + ["irregular"] * (N_POLYGONS_PER_TIER // 4))
    targets = [draw_targets(rng, shape) for shape in plan]
    for tier in TIER_ORDER:
        for i, (shape, (af, ar)) in enumerate(zip(plan, targets)):
            p = None
            for _ in range(40):                      # small many-vertex polygons need many draws on the integer grid
                p, knobs, why = make_one(rng, shape, tier, af * span(tier) ** 2, ar)
                if p is not None: break
            if p is None:
                raise RuntimeError(f"corpus slot {tier}/{shape}/{i} failed: {why}")
            corpus.append({"object_id": f"exp2_{tier}_{shape}_{i+1:03d}", "tier": tier,
                           "shape_type": shape, "poly": p, "knobs": knobs})
    return corpus

## [CELL 4] Pipeline builders

* **pass-through**: every step states the tool and its arguments; the model
  forwards the object.
* **property-conditioned**, two mechanisms as in Table 2. *Threshold*: `If the
  <gate> of {o0} is greater than <thr>, call scale({o0}, 0.85). Otherwise call
  buffer({o0}, <d>).` with the threshold a recorded margin above or below the
  true value, branches balanced. *Derived argument*: `Call buffer({o0}, d)
  where d is 5% of the perimeter of {o0}` (and translate by 50 % of the
  width), signs alternating along the pipeline so the object stays bounded,
  scored at 5 % relative tolerance.
* **multi-object**: `Call <tool>(...) on whichever of {o0}, {o1}, {o2} has the
  largest <gate>.` Three polygons from the same tier; the gap between the
  winner and the runner-up is a recorded margin.

Every gated step records `decision_margin`, `margin_band` and
`allowed_tools`. A call outside `allowed_tools` is a deferral.

In [ ]:
def _diag(p):
    a, b, c, d = p.bounds
    return math.dist((a, b), (c, d))

def _try(tool, ops, args):
    try:
        out = apply_op(tool, ops, args, strict=True)
    except Exception:
        return None
    return out if is_terminal(tool) or is_runtime_valid(out) else None

def _noop(a, b):
    try:
        return b.normalize().equals_exact(a.normalize(), 1e-6)
    except Exception:
        return False

def _overlap(base, other, rng, frac=0.35):
    o = affinity.translate(other, xoff=base.centroid.x - other.centroid.x, yoff=base.centroid.y - other.centroid.y)
    s = _diag(base) / max(_diag(o), 1e-9)
    o = affinity.scale(o, xfact=s, yfact=s, origin="centroid")
    return affinity.translate(o, xoff=frac * _diag(base), yoff=frac * 0.4 * _diag(base))

ARGFREE = ["convex_hull", "bounding_box"]
ARGED = {"buffer":    lambda r, g: [round(0.05 * _diag(g), 3)],
         "simplify":  lambda r, g: [round(0.05 * _diag(g), 3)],
         "rotate":    lambda r, g: [r.choice([30, 45, 90, 180])],
         "translate": lambda r, g: [r.choice([5, 10, 20, -10]), r.choice([5, 10, -15])],
         "scale":     lambda r, g: [r.choice([0.5, 1.5, 2.0])]}


class Builder:
    def __init__(self):
        self.n = 0; self.store = {}; self.initial = []
    def add(self, poly, oid=None):
        self.n += 1; h = f"polygon_{self.n}"; poly = snap(poly); self.store[h] = poly
        self.initial.append({"handle": h, "object_id": oid, "wkt": to_wkt(poly)}); return h
    def new(self):
        self.n += 1; return f"polygon_{self.n}"


def _step(i, kind, instr, vis, tool, ops, args, out, allowed, gate=None, margin=None, band=None,
          noop=False, mech=None, branch=None):
    return {"index": i, "kind": kind, "instruction": instr, "visible_handles": vis,
            "correct_tool": tool, "correct_operands": ops, "correct_args": args,
            "output_handle": out, "is_terminal": is_terminal(tool), "is_noop": bool(noop),
            "allowed_tools": allowed, "gate_property": gate, "decision_margin": margin,
            "margin_band": band, "mechanism": mech, "branch": branch}


def build_pass_through(rng, start, oid, depth, pool):
    b = Builder(); cur = b.add(start, oid); steps = []
    for i in range(depth):
        last = (i == depth - 1)
        if last and depth >= 2 and rng.random() < 0.25:
            tool = rng.choice(TERMINAL_OPS)
            if n_operands(tool) == 2:
                oh = b.add(_overlap(b.store[cur], pool[rng.randrange(len(pool))], rng))
                steps.append(_step(i, "explicit", f"Call {tool} on {{o0}} and {{o1}}.", [cur, oh], tool, [cur, oh], [], None, [tool]))
            else:
                steps.append(_step(i, "explicit", f"Call {tool} on {{o0}}.", [cur], tool, [cur], [], None, [tool]))
            break
        if rng.random() < 0.20:
            tool = rng.choice(["intersection", "union"])
            cand = snap(_overlap(b.store[cur], pool[rng.randrange(len(pool))], rng))
            res = _try(tool, [b.store[cur], cand], [])
            if res is not None:
                oh = b.add(cand); out = b.new()
                steps.append(_step(i, "explicit", f"Call {tool} on {{o0}} and {{o1}}.", [cur, oh], tool, [cur, oh], [], out, [tool],
                                   noop=_noop(b.store[cur], res)))
                b.store[out] = res; cur = out; continue
        if rng.random() < 0.3:
            tool, args = rng.choice(ARGFREE), []; instr = f"Call {tool} on {{o0}}."
        else:
            tool = rng.choice(list(ARGED)); args = ARGED[tool](rng, b.store[cur])
            instr = f"Call {tool} on {{o0}} with arguments ({', '.join(str(a) for a in args)})."
        res = _try(tool, [b.store[cur]], args)
        if res is None:
            tool, args = "translate", [10, 10]; instr = "Call translate on {o0} with arguments (10, 10)."
            res = _try(tool, [b.store[cur]], args)
        out = b.new()
        steps.append(_step(i, "explicit", instr, [cur], tool, [cur], args, out, [tool], noop=_noop(b.store[cur], res)))
        b.store[out] = res; cur = out
    return b, steps, cur


def build_property_conditioned(rng, start, oid, depth, gate, band, branches):
    b = Builder(); cur = b.add(start, oid); steps = []
    lo, hi = MARGIN_BANDS[band]
    for i in range(depth):
        q = properties(b.store[cur]); val = GATE_VALUE[gate](q)
        margin = rng.uniform(lo, hi); above = branches[i]
        thr = round(val * (1 - margin) if above else val * (1 + margin), 3)
        d = round(0.05 * _diag(b.store[cur]), 3)
        tool, args = ("scale", [PC_SCALE]) if above else ("buffer", [d])
        instr = (f"If the {GATE_PHRASE[gate]} of {{o0}} is greater than {thr}, call scale({{o0}}, {PC_SCALE}). "
                 f"Otherwise call buffer({{o0}}, {d}). Answer with exactly one of these two calls.")
        res = _try(tool, [b.store[cur]], args)
        if res is None:
            return None, None, None                  # caller retries with another start polygon
        out = b.new()
        steps.append(_step(i, "gated", instr, [cur], tool, [cur], args, out, ["scale", "buffer"], gate=gate,
                           margin=round(margin, 4), band=band, noop=_noop(b.store[cur], res), mech="threshold",
                           branch="above" if above else "below"))
        b.store[out] = res; cur = out
    return b, steps, cur


def build_property_derived(rng, start, oid, depth, gate):
    """Each step's argument is a fraction of a latent property of the current
    object (Table 2, second property-conditioned mechanism)."""
    b = Builder(); cur = b.add(start, oid); steps = []
    g, tool, fracs, phrase_pos, phrase_neg = next(t for t in PC_DERIVED if t[0] == gate)
    for i in range(depth):
        q = properties(b.store[cur]); val = GATE_VALUE[g](q)
        frac = fracs[i % 2]; phrase = phrase_pos if i % 2 == 0 else phrase_neg   # wording follows the STEP INDEX, not the sign
        d = round(frac * val, 3)
        neg = " (a negative number)" if frac < 0 else ""
        if tool == "buffer":
            args = [d]
            instr = f"Call buffer({{o0}}, d) where d is {phrase} of {{o0}}{neg}. Give d as a number."
        elif tool == "rotate":
            args = [d]
            instr = f"Call rotate({{o0}}, angle) where angle is {phrase} of {{o0}}. Give angle as a number."
        else:
            args = [d, 0]
            instr = f"Call translate({{o0}}, dx, 0) where dx is {phrase} of {{o0}}{neg}. Give dx as a number."
        res = _try(tool, [b.store[cur]], args)
        if res is None:
            return None, None, None
        out = b.new()
        steps.append(_step(i, "gated", instr, [cur], tool, [cur], args, out, [tool], gate=g, margin=None,
                           band="derived", noop=_noop(b.store[cur], res), mech="derived", branch=None))
        b.store[out] = res; cur = out
    return b, steps, cur


def build_property_boolean(rng, start, oid, depth):
    """Convexity gate (Table 2): `If {o0} is convex, call scale; otherwise buffer`."""
    b = Builder(); cur = b.add(start, oid); steps = []
    # Both branches are integer translations, which are EXACT on the integer grid,
    # so convexity cannot drift along the pipeline and the branch balance stays
    # exactly 50/50.  (scale / buffer / rotate all re-round the coordinates, and
    # that rounding flips convexity on ~18% of steps, which would bias the gate.)
    for i in range(depth):
        q = properties(b.store[cur]); cvx = bool(q["convex"])
        args = [20, 0] if cvx else [0, 20]
        instr = ("If {o0} is convex, call translate({o0}, 20, 0). "
                 "Otherwise call translate({o0}, 0, 20). Answer with exactly one of these two calls.")
        res = _try("translate", [b.store[cur]], args)
        if res is None:
            return None, None, None
        out = b.new()
        steps.append(_step(i, "gated", instr, [cur], "translate", [cur], args, out, ["translate"], gate="convex",
                           margin=None, band="boolean", noop=_noop(b.store[cur], res), mech="boolean",
                           branch="convex" if cvx else "nonconvex"))
        b.store[out] = res; cur = out
    return b, steps, cur


def _mo_margin(vals):
    s = sorted(vals, reverse=True)
    return (s[0] - s[1]) / s[0] if s[0] > 0 else 0.0

def _in_band(m, gate, band):
    lo, hi = (VC_BANDS if gate == "vertex_count" else MARGIN_BANDS)[band]
    return lo <= m <= hi

def find_group(rng, polys, gate, band, n=MO_N_OBJECTS, tries=400):
    """n polygons from one tier whose winner/runner-up gap is in the band and
    whose third value is at or below the runner-up."""
    vals = [GATE_VALUE[gate](properties(p)) for p in polys]
    idx = list(range(len(polys)))
    for _ in range(tries):
        g = rng.sample(idx, n)
        v = [vals[i] for i in g]
        if len(set(v)) < n and gate != "vertex_count":
            continue
        s = sorted(v, reverse=True)
        if gate == "vertex_count" and s[0] == s[1]:
            continue
        if _in_band(_mo_margin(v), gate, band):
            return g
    return None

MO_CANDIDATES = [("scale", [0.5]), ("scale", [0.75]), ("scale", [1.5]), ("scale", [2.0]),
                 ("rotate", [90]), ("rotate", [45]), ("translate", [20, -10]), ("translate", [-15, 10])]
# ---------------------------------------------------------------------------
# Multi-object versions of the three property-conditioned mechanisms.
# Table 2 defines multi-object as "selecting among OR COMPARING multiple serialized
# objects".  Selection is one form; a condition that spans two objects, or an
# argument taken from one object and applied to another, are the others.  Giving
# multi-object all four mechanisms makes the design a 2 x 4 factorial
# ({1 object, 4 objects} x {threshold, boolean, derived, selection}) minus the one
# structurally empty cell (1 object x selection), so the category means are
# computed over the SAME mechanism and band mix and become comparable.
# Roles (A: condition/source, B: condition, C: acted on, D: distractor) are
# reshuffled every step so no object is special across the pipeline.
# ---------------------------------------------------------------------------

def _roles(rng, handles, k):
    vis = handles[:]; rng.shuffle(vis)
    return vis, vis[:k]

def build_mo_threshold(rng, polys, ids, depth, gate, band, branches):
    """If <gate>(A) > T1 and <gate>(B) > T2, call scale(C, s). Otherwise call buffer(C, d).
    Two-way decision (floor 0.50) that needs the property of TWO objects."""
    b = Builder(); handles = [b.add(p, o) for p, o in zip(polys, ids)]; steps = []
    lo, hi = MARGIN_BANDS[band]
    for i in range(depth):
        vis, (A, B, C) = _roles(rng, handles, 3)
        vA = GATE_VALUE[gate](properties(b.store[A])); vB = GATE_VALUE[gate](properties(b.store[B]))
        m1, m2 = rng.uniform(lo, hi), rng.uniform(lo, hi)
        both = branches[i]
        if both:
            T1, T2 = vA * (1 - m1), vB * (1 - m2)
        else:                                     # exactly one condition fails, chosen at random
            if rng.random() < 0.5: T1, T2 = vA * (1 + m1), vB * (1 - m2)
            else:                  T1, T2 = vA * (1 - m1), vB * (1 + m2)
        T1, T2 = round(T1, 3), round(T2, 3)
        d = round(0.05 * _diag(b.store[C]), 3)
        tool, args = ("scale", [PC_SCALE]) if both else ("buffer", [d])
        iA, iB, iC = vis.index(A), vis.index(B), vis.index(C)
        instr = (f"If the {GATE_PHRASE[gate]} of {{o{iA}}} is greater than {T1} and the {GATE_PHRASE[gate]} of "
                 f"{{o{iB}}} is greater than {T2}, call scale({{o{iC}}}, {PC_SCALE}). Otherwise call "
                 f"buffer({{o{iC}}}, {d}). Answer with exactly one of these two calls.")
        res = _try(tool, [b.store[C]], args)
        if res is None: return None, None, None
        out = b.new()
        steps.append(_step(i, "gated", instr, vis, tool, [C], args, out, ["scale", "buffer"], gate=gate,
                           margin=round(min(m1, m2), 4), band=band, noop=_noop(b.store[C], res), mech="threshold",
                           branch="above" if both else "below"))
        b.store[out] = res; handles = [out if h == C else h for h in handles]
    return b, steps, handles

def build_mo_boolean(rng, polys, ids, depth, branches):
    """If A is convex and B is convex, call translate(C, 20, 0). Otherwise translate(C, 0, 20).
    Needs a convexity test on TWO objects.  Requires the group to contain at least
    two convex and one non-convex polygon so both branches are constructible."""
    b = Builder(); handles = [b.add(p, o) for p, o in zip(polys, ids)]; steps = []
    for i in range(depth):
        conv = [h for h in handles if properties(b.store[h])["convex"]]
        nonc = [h for h in handles if not properties(b.store[h])["convex"]]
        if len(conv) < 2 or len(nonc) < 1: return None, None, None
        both = branches[i]
        if both: A, B = rng.sample(conv, 2)
        else:
            A = rng.choice(nonc); B = rng.choice([h for h in handles if h != A])
            if rng.random() < 0.5: A, B = B, A
        C = rng.choice([h for h in handles if h not in (A, B)])
        vis = handles[:]; rng.shuffle(vis)
        iA, iB, iC = vis.index(A), vis.index(B), vis.index(C)
        args = [20, 0] if both else [0, 20]
        instr = (f"If {{o{iA}}} is convex and {{o{iB}}} is convex, call translate({{o{iC}}}, 20, 0). "
                 f"Otherwise call translate({{o{iC}}}, 0, 20). Answer with exactly one of these two calls.")
        res = _try("translate", [b.store[C]], args)
        if res is None: return None, None, None
        out = b.new()
        steps.append(_step(i, "gated", instr, vis, "translate", [C], args, out, ["translate"], gate="convex",
                           margin=None, band="boolean", noop=False, mech="boolean", branch="both" if both else "not_both"))
        b.store[out] = res; handles = [out if h == C else h for h in handles]
    return b, steps, handles

MO_DERIVED = [("perimeter", "buffer", (0.05, -0.04), "5% of the perimeter", "minus 4% of the perimeter"),
              ("aspect_ratio", "rotate", (30.0, 25.0),
               "30 times the aspect ratio (bounding-box width divided by bounding-box height)",
               "25 times the aspect ratio (bounding-box width divided by bounding-box height)")]

def build_mo_derived(rng, polys, ids, depth, gate):
    """Call buffer(B, d) where d is 5% of the perimeter of A.  The argument is derived
    from one object and applied to a different one."""
    b = Builder(); handles = [b.add(p, o) for p, o in zip(polys, ids)]; steps = []
    g, tool, fracs, ph_pos, ph_neg = next(t for t in MO_DERIVED if t[0] == gate)
    for i in range(depth):
        vis, (A, B) = _roles(rng, handles, 2)
        frac = fracs[i % 2]; phrase = ph_pos if i % 2 == 0 else ph_neg
        d = round(frac * GATE_VALUE[g](properties(b.store[A])), 3); args = [d]
        iA, iB = vis.index(A), vis.index(B); neg = " (a negative number)" if frac < 0 else ""
        if tool == "buffer":
            instr = f"Call buffer({{o{iB}}}, d) where d is {phrase} of {{o{iA}}}{neg}. Give d as a number."
        else:
            instr = f"Call rotate({{o{iB}}}, angle) where angle is {phrase} of {{o{iA}}}. Give angle as a number."
        res = _try(tool, [b.store[B]], args)
        if res is None: return None, None, None
        out = b.new()
        steps.append(_step(i, "gated", instr, vis, tool, [B], args, out, [tool], gate=g, margin=None,
                           band="derived", noop=_noop(b.store[B], res), mech="derived"))
        b.store[out] = res; handles = [out if h == B else h for h in handles]
    return b, steps, handles


def build_multi_object(rng, polys, ids, depth, gate, band):
    b = Builder(); handles = [b.add(p, o) for p, o in zip(polys, ids)]; steps = []
    for i in range(depth):
        vis = handles[:]; rng.shuffle(vis)
        vals = {h: GATE_VALUE[gate](properties(b.store[h])) for h in vis}
        win = max(vis, key=lambda h: vals[h]); margin = _mo_margin(list(vals.values()))
        # choose the operation so that the NEXT configuration stays in band when possible
        cands = MO_CANDIDATES[:]; rng.shuffle(cands); chosen = None
        for tool, args in cands:
            res = _try(tool, [b.store[win]], args)
            if res is None: continue
            nxt = dict(vals); nxt[win] = GATE_VALUE[gate](properties(res))
            if i == depth - 1 or _in_band(_mo_margin(list(nxt.values())), gate, band):
                chosen = (tool, args, res); break
            if chosen is None: chosen = (tool, args, res)
        if chosen is None:
            res = _try("translate", [b.store[win]], [20, -10]); chosen = ("translate", [20, -10], res)
        tool, args, res = chosen
        out = b.new()
        names = ", ".join("{o%d}" % j for j in range(len(vis) - 1)) + " and {o%d}" % (len(vis) - 1)
        sup = "" if gate == "centroid_y" else "largest "
        instr = (f"Call {tool} with arguments ({', '.join(str(a) for a in args)}) on whichever of "
                 f"{names} has the {sup}{GATE_PHRASE[gate]}.")
        steps.append(_step(i, "gated", instr, vis, tool, [win], args, out, [tool], gate=gate,
                           margin=round(margin, 4), band=band, noop=_noop(b.store[win], res), mech="selection"))
        b.store[out] = res
        handles = [out if h == win else h for h in handles]
    return b, steps, handles

## [CELL 5] Build the corpus and the pipelines

Stratified on category, depth, tier, gate and margin band.

In [ ]:
def generate():
    rng = random.Random(SEED)
    corpus = build_corpus(rng)
    POLY = {c["object_id"]: c["poly"] for c in corpus}; META = {c["object_id"]: c for c in corpus}
    by_tier = {t: [c["object_id"] for c in corpus if c["tier"] == t] for t in TIER_ORDER}
    pool = [c["poly"] for c in corpus]
    pls = []

    def finish(pid, cat, gate, band, depth, b, steps, final_handles, sids):
        ef = None
        if steps and not steps[-1]["is_terminal"] and cat != "multi_object":
            fp = b.store[final_handles]; q = properties(fp)
            ef = {"handle": final_handles, "wkt": to_wkt(fp), "area": q["area"], "perimeter": q["perimeter"], "vertex_count": q["vertex_count"]}
        pls.append({"pipeline_id": pid, "category": cat, "gate_property": gate, "gate_locality": GATE_LOCALITY.get(gate),
                    "margin_band": band, "depth": depth,
                    "tier": META[sids[0]]["tier"], "shape_type": META[sids[0]]["shape_type"],
                    "start_object_ids": sids, "initial_objects": b.initial, "steps": steps, "expected_final": ef})

    # pass-through: tiers rotate
    for depth in DEPTHS:
        for n in range(PER_CELL["pass_through"]):
            t = TIER_ORDER[n % 3]; oid = rng.choice(by_tier[t])
            b, s, fh = build_pass_through(rng, POLY[oid], oid, depth, pool)
            finish(f"pt__none__none__k{depth}__{n:03d}", "pass_through", None, None, depth, b, s, fh, [oid])

    # property-conditioned: 6 threshold combos (gate x band) + 2 derived-argument combos,
    # tiers rotate, branches balanced per cell
    # 15 combos: 8 global threshold (4 gates x tight/wide), 1 global boolean, 2 global derived, 4 local controls.
    combos = ([(g, bnd) for g in ["area", "perimeter", "aspect_ratio", "edge_length_variance"] for bnd in ["tight", "wide"]]
              + [("convex", "boolean")]
              + [("perimeter", "derived"), ("aspect_ratio", "derived")]
              + [("width", "tight"), ("width", "wide"), ("vertex_count", "tight"), ("vertex_count", "wide")])
    assert len(combos) == 15
    conv_toggle = {}
    for depth in DEPTHS:
        for n in range(PER_CELL["property_conditioned"]):
            t = TIER_ORDER[n % 3]; gate, band = combos[(n // 3) % len(combos)]
            for _ in range(60):
                if band == "boolean":
                    # alternate convex / non-convex start objects on a single running counter so the
                    # gate's majority baseline is exactly 50% across the whole corpus
                    conv_toggle["n"] = conv_toggle.get("n", 0) + 1
                    want = (conv_toggle["n"] % 2 == 1)
                    pool_t = [o for o in by_tier[t] if bool(properties(POLY[o])["convex"]) == want]
                    oid = rng.choice(pool_t)
                    b, s, fh = build_property_boolean(rng, POLY[oid], oid, depth)
                elif band == "derived":
                    oid = rng.choice(by_tier[t])
                    b, s, fh = build_property_derived(rng, POLY[oid], oid, depth, gate)
                else:
                    oid = rng.choice(by_tier[t])
                    branches = [True] * (depth // 2) + [False] * (depth - depth // 2)
                    if depth % 2 == 1 and n % 2 == 1: branches[-1] = True
                    rng.shuffle(branches)
                    b, s, fh = build_property_conditioned(rng, POLY[oid], oid, depth, gate, band, branches)
                if b is not None: break
            else:
                raise RuntimeError(f"no valid property-conditioned pipeline for {gate}/{band}/{t}/k{depth}")
            finish(f"pc__{gate}__{band}__k{depth}__{n:03d}", "property_conditioned", gate, band, depth, b, s, fh, [oid])

    # multi-object: (gate x band) cycle, tiers rotate, all three objects from one tier
    # 21 combos: the SAME 8 threshold + 1 boolean + 2 derived cells as property-conditioned,
    # plus 8 global selection cells (multi-object's own mechanism) and 2 local selection controls.
    combos = ([(g, bnd, "threshold") for g in ["area", "perimeter", "aspect_ratio", "edge_length_variance"] for bnd in ["tight", "wide"]]
              + [("convex", "boolean", "boolean")]
              + [("perimeter", "derived", "derived"), ("aspect_ratio", "derived", "derived")]
              + [(g, bnd, "selection") for g in ["area", "perimeter", "aspect_ratio", "edge_length_variance"] for bnd in ["tight", "wide"]]
              + [("width", "tight", "selection"), ("vertex_count", "tight", "selection")])
    assert len(combos) == 21
    for depth in DEPTHS:
        for n in range(PER_CELL["multi_object"]):
            t = TIER_ORDER[n % 3]; gate, band, mech = combos[(n // 3) % len(combos)]
            ids = by_tier[t]
            branches = [True] * (depth // 2) + [False] * (depth - depth // 2)
            if depth % 2 == 1 and n % 2 == 1: branches[-1] = True
            rng.shuffle(branches)
            b = None
            for _attempt in range(80):
                if mech == "selection":
                    grp = find_group(rng, [POLY[o] for o in ids], gate, band)
                    if grp is None: break
                    sids = [ids[i] for i in grp]
                    b, s, fh = build_multi_object(rng, [POLY[o] for o in sids], sids, depth, gate, band)
                elif mech == "boolean":
                    cv = [o for o in ids if properties(POLY[o])["convex"]]; nc = [o for o in ids if not properties(POLY[o])["convex"]]
                    sids = rng.sample(cv, 2) + rng.sample(nc, 2); rng.shuffle(sids)
                    b, s, fh = build_mo_boolean(rng, [POLY[o] for o in sids], sids, depth, branches)
                else:
                    sids = rng.sample(ids, MO_N_OBJECTS)
                    if mech == "threshold":
                        b, s, fh = build_mo_threshold(rng, [POLY[o] for o in sids], sids, depth, gate, band, branches)
                    else:
                        b, s, fh = build_mo_derived(rng, [POLY[o] for o in sids], sids, depth, gate)
                if b is not None: break
            if b is None:
                raise RuntimeError(f"multi-object {mech} failed for {gate}/{band}/{t}/k{depth}")
            finish(f"mo__{gate}__{band}__{mech}__k{depth}__{n:03d}", "multi_object", gate, band, depth, b, s, fh, sids)
    return pls, corpus


pipelines, corpus = generate()
n_steps = sum(len(p["steps"]) for p in pipelines)
print(f"corpus    {len(corpus):,} polygons")
print(f"pipelines {len(pipelines):,}   steps {n_steps:,}")
print(f"API calls per model = steps x 4 conditions x 2 modes = {n_steps*8:,}  (upper bound; cascading aborts early)\n")
for cat in PER_CELL:
    s = [p for p in pipelines if p["category"] == cat]
    print(f"  {cat:22s} {len(s):5,} pipelines  {sum(len(x['steps']) for x in s):6,} steps")

## [CELL 6] Corpus design check

In [ ]:
fails = []
def check(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}{('  ' + detail) if detail else ''}")
    if not ok: fails.append(name)

def vals(fn, t):
    return np.array([fn(c) for c in corpus if c["tier"] == t], float)

print("MATCHED across tiers")
for prop in ["area", "width", "perimeter"]:
    fn = lambda c, prop=prop: properties(c["poly"])[prop]
    d = max(ks_2samp(vals(fn, "simple"), vals(fn, t)).statistic for t in ("medium", "hard"))
    check(f"{prop} matched (D < 0.20)", d < 0.20, f"max D {d:.3f}")
print("SEPARATED across tiers")
for name, fn in [("vertex_count", lambda c: properties(c["poly"])["vertex_count"]), ("wkt_length", lambda c: len(to_wkt(c["poly"])))]:
    d = min(ks_2samp(vals(fn, "simple"), vals(fn, "medium")).statistic, ks_2samp(vals(fn, "medium"), vals(fn, "hard")).statistic)
    check(f"{name} separated (D > 0.85)", d > 0.85, f"min D {d:.3f}")   # protocol ranges share the value 20 between medium and hard
for t in TIER_ORDER:
    cv = sum(properties(c["poly"])["convex"] for c in corpus if c["tier"] == t)
    check(f"{t}: convex 50/50", cv == N_POLYGONS_PER_TIER // 2, f"{cv}/{N_POLYGONS_PER_TIER}")

## [CELL 7] Pre-registered prediction of Table 4

Inputs are **Experiment 1 numbers and chance floors only**. `EXP1_P` is the
fraction of Exp1 answers within 5 % of the truth for each gate property,
pooled over models and tiers. Paste the values from the new Exp1 Phase 3 run
here before running Phase 2; the defaults are the v1 Exp1 numbers. The three
`ASSUME_*` constants are stated assumptions, not observations.

In [ ]:
EXP1_P = {"area": 0.05, "perimeter": 0.15, "aspect_ratio": 0.08, "edge_length_variance": 0.03,
          "centroid_y": 0.45, "width": 0.95, "vertex_count": 0.55, "convex": 0.50}   # <- replace from Exp1 v3 Phase 3
ASSUME_COPY_FIDELITY = 0.95     # raw pass-through: a correct call is a faithful WKT copy
ASSUME_READ_STATED   = 0.95     # a stated value is read correctly (augmented, handle+summary)
ASSUME_COMPOSE_STATED = 0.85    # two stated values are combined correctly (aspect ratio = width / height)
ASSUME_RANK_STATED   = 0.90     # the stated values of MO_N_OBJECTS objects are ranked correctly
COMPOSITE_GATES = {"aspect_ratio"}   # not pre-computed in the summary; must be composed from width and height

SIGMA = {g: 0.05 / norm.ppf(0.5 + max(min(p, 0.999), 0.001) / 2) for g, p in EXP1_P.items()}
# half-normal relative error e ~ N(0, sigma): P(|e| <= 0.05) = EXP1_P

def p_threshold(gate, margin):
    return float(norm.cdf(margin / SIGMA[gate]))          # correct side of a threshold at the given margin

def p_select(gate, margin, n=MO_N_OBJECTS, rng=np.random.default_rng(0), k=4000):
    s = SIGMA[gate]
    v = np.array([1.0, 1.0 - margin] + [1.0 - margin - 0.2] * (n - 2))
    est = v * (1 + rng.normal(0, s, size=(k, n)))
    return float((est.argmax(axis=1) == 0).mean())

def predict(pls):
    rows = []
    for cat in PER_CELL:
        for loc in (["all"] if cat == "pass_through" else ["global", "local"]):
          for band in ([None] if cat == "pass_through" else (list(MARGIN_BANDS) + (["derived", "boolean"] if cat == "property_conditioned" else []))):
            raw, hnd, hsm, aug = [], [], [], []
            for p in pls:
                if p["category"] != cat or p["margin_band"] != band: continue
                if loc != "all" and p["gate_locality"] != loc: continue
                for s in p["steps"]:
                    if cat == "pass_through":
                        raw.append(ASSUME_COPY_FIDELITY); aug.append(ASSUME_COPY_FIDELITY); hnd.append(1.0); hsm.append(1.0)
                    elif s["mechanism"] == "threshold":
                        pt = p_threshold(s["gate_property"], s["decision_margin"])
                        raw.append(pt if len(s["visible_handles"]) == 1 else 0.5 + (pt - 0.5) * pt)   # two-object conjunction
                        st = ASSUME_COMPOSE_STATED if s["gate_property"] in COMPOSITE_GATES else ASSUME_READ_STATED
                        aug.append(st); hnd.append(0.5); hsm.append(st)
                    elif s["mechanism"] == "derived":
                        raw.append(EXP1_P[s["gate_property"]])          # argument within 5% of a computed value
                        st = ASSUME_COMPOSE_STATED if s["gate_property"] in COMPOSITE_GATES else ASSUME_READ_STATED
                        aug.append(st); hnd.append(0.0); hsm.append(st)
                    elif s["mechanism"] == "boolean":
                        raw.append(EXP1_P.get("convex", 0.5))            # Exp1: models answer the majority class
                        aug.append(ASSUME_READ_STATED); hnd.append(0.5); hsm.append(ASSUME_READ_STATED)
                    else:
                        raw.append(p_select(s["gate_property"], max(s["decision_margin"], 1e-4)))
                        st = ASSUME_RANK_STATED * (ASSUME_COMPOSE_STATED if s["gate_property"] in COMPOSITE_GATES else 1.0)
                        aug.append(st); hnd.append(1.0 / MO_N_OBJECTS); hsm.append(st)
            if raw:
                rows.append(dict(category=cat, locality=loc, band=band, raw=float(np.mean(raw)), augmented=float(np.mean(aug)),
                                 handle=float(np.mean(hnd)), handle_sum=float(np.mean(hsm)), n=len(raw)))
    return rows

rows = predict(pipelines)
print("PREDICTED TABLE 4 (per-step accuracy, oracle mode).  HEADLINE = global gates; local gates are the control.")
print(f"  {'category':22s} {'gates':7s} {'band':8s} {'raw':>7s} {'augm.':>7s} {'handle':>7s} {'hnd+sum':>8s} {'n':>6s}")
for r in rows:
    print(f"  {r['category']:22s} {r['locality']:7s} {str(r['band']):8s} {r['raw']:7.3f} {r['augmented']:7.3f} {r['handle']:7.3f} {r['handle_sum']:8.3f} {r['n']:6,}")
print("\n  handle column = the chance floor (no property information).  Under handle and handle_sum the runtime also\n"
      "  answers get_object(id) with the WKT, so a model that fetches the text can do no better than the raw column:\n"
      "  the prediction for handle on gated steps is  chance <= handle <= raw.")
print("\nPredicted orderings:")
print("  columns: pass-through >= property-conditioned >= multi-object in every condition")
print("  rows:    raw <= augmented ~= handle_sum;  handle = chance floor on gated steps")
print("  raw vs handle on gated steps: raw >= handle in the wide band, ~equal in the tight band (this is a prediction; report either way)")

## [CELL 8] Structural validation

Every answer key is recomputed from the corpus and compared.

In [ ]:
POLY = {c["object_id"]: c["poly"] for c in corpus}
def rebuild(p):
    return {o["handle"]: shapely_wkt.loads(o["wkt"]) for o in p["initial_objects"]}

bad_thr = bad_sel = bad_der = bad_bool = bad_exec = c_thr = c_sel = c_der = c_bool = 0
for p in pipelines:
    store = rebuild(p)
    for s in p["steps"]:
        ops = [store[h] for h in s["correct_operands"]]
        if s["mechanism"] == "threshold":
            c_thr += 1
            ths = [float(x) for x in re.findall(r"greater than ([\d.\-]+)", s["instruction"])]
            objs = re.findall(r"of \{o(\d)\} is greater than", s["instruction"])
            if len(ths) == 1:                                            # single-object
                val = GATE_VALUE[s["gate_property"]](properties(ops[0]))
                if ("scale" if val > ths[0] else "buffer") != s["correct_tool"]: bad_thr += 1
            else:                                                        # two-object conjunction
                hs = [s["visible_handles"][int(k)] for k in objs]
                vals = [GATE_VALUE[s["gate_property"]](properties(store[h])) for h in hs]
                both = all(v > T for v, T in zip(vals, ths))
                if ("scale" if both else "buffer") != s["correct_tool"]: bad_thr += 1
            if s["correct_tool"] == "scale" and s["correct_args"] != [PC_SCALE]: bad_thr += 1
        if s["mechanism"] == "boolean":
            c_bool += 1
            objs = re.findall(r"\{o(\d)\} is convex", s["instruction"])
            hs = [s["visible_handles"][int(k)] for k in objs] if objs else s["correct_operands"]
            both = all(properties(store[h])["convex"] for h in hs)
            if ([20, 0] if both else [0, 20]) != s["correct_args"]: bad_bool += 1
        if s["mechanism"] == "derived":
            c_der += 1
            g, tool, fracs, _, _ = next(t for t in PC_DERIVED if t[0] == s["gate_property"])
            src_m = re.search(r"of \{o(\d)\}(?: \(a negative number\))?\. Give", s["instruction"])
            src_h = s["visible_handles"][int(src_m.group(1))] if src_m and len(s["visible_handles"]) > 1 else s["correct_operands"][0]
            want = round(fracs[s["index"] % 2] * GATE_VALUE[g](properties(store[src_h])), 3)
            if s["correct_tool"] != tool: bad_der += 1
            if s["correct_tool"] != tool or abs(s["correct_args"][0] - want) > 1e-6: bad_der += 1
        if s["mechanism"] == "selection":
            c_sel += 1
            v = [GATE_VALUE[s["gate_property"]](properties(store[h])) for h in s["visible_handles"]]
            if s["visible_handles"][int(np.argmax(v))] != s["correct_operands"][0]: bad_sel += 1
        try: r_ = apply_op(s["correct_tool"], ops, s["correct_args"])
        except Exception: bad_exec += 1; continue
        if s["output_handle"]: store[s["output_handle"]] = r_
check("threshold keys", bad_thr == 0, f"{c_thr:,} checked")
check("selection winners", bad_sel == 0, f"{c_sel:,} checked")
check("derived-argument keys", bad_der == 0, f"{c_der:,} checked")
import re as _re
bad_txt = 0
for p in pipelines:
    for s in p["steps"]:
        if s["mechanism"] not in ("derived", "selection_derived"): continue
        m = _re.search(r"(?:is|angle is) ([\d.]+)", s["instruction"])
        if not m: continue
        stated = float(m.group(1))
        src = PC_DERIVED if s["mechanism"] == "derived" else MO_DERIVED
        g, tool, fracs, _, _ = next(t for t in src if t[0] == s["gate_property"])
        used = abs(fracs[s["index"] % 2]) * (100 if abs(fracs[s["index"] % 2]) < 1 else 1)
        if abs(stated - used) > 1e-6: bad_txt += 1
check("derived instruction text matches the key", bad_txt == 0, f"{bad_txt} mismatched")
check("convexity-gate keys", bad_bool == 0, f"{c_bool:,} checked")
check("all keys executable", bad_exec == 0)

bw = chk = 0
for p in pipelines:
    if not p["expected_final"]: continue
    chk += 1; store = rebuild(p)
    for s in p["steps"]:
        r_ = apply_op(s["correct_tool"], [store[h] for h in s["correct_operands"]], s["correct_args"])
        if s["output_handle"]: store[s["output_handle"]] = r_
    ef = p["expected_final"]
    if not store[ef["handle"]].normalize().equals_exact(shapely_wkt.loads(ef["wkt"]).normalize(), 1e-9): bw += 1
check("replay reproduces expected_final", bw == 0, f"{chk:,} pipelines")
check("terminal ops only last", not any(s["is_terminal"] and i != len(p["steps"]) - 1 for p in pipelines for i, s in enumerate(p["steps"])))
check("operand counts match signatures", all(len(s["correct_operands"]) == n_operands(s["correct_tool"]) for p in pipelines for s in p["steps"]))
check("correct tool is always admissible", all(s["correct_tool"] in s["allowed_tools"] for p in pipelines for s in p["steps"]))
check("no literal handle in any template", not any(re.search(r"polygon_\d", s["instruction"]) for p in pipelines for s in p["steps"]))

for cat in ("property_conditioned", "multi_object"):
    for g in ["area", "perimeter", "aspect_ratio", "edge_length_variance", "width", "vertex_count"]:
        sel = [s for p in pipelines if p["category"] == cat and p["gate_property"] == g for s in p["steps"] if s["mechanism"] == "threshold"]
        if not sel: continue
        sc = sum(s["correct_tool"] == "scale" for s in sel)
        check(f"{cat}: threshold branch balance {g}", max(sc, len(sel) - sc) / len(sel) <= 0.56, f"majority {100*max(sc, len(sel)-sc)/len(sel):.1f}%")
    sel = [s for p in pipelines if p["category"] == cat and p["gate_property"] == "convex" for s in p["steps"]]
    if sel:
        sc = sum(s["correct_args"] == [20, 0] for s in sel)
        check(f"{cat}: boolean branch balance", max(sc, len(sel) - sc) / len(sel) <= 0.56, f"majority {100*max(sc, len(sel)-sc)/len(sel):.1f}%")
pos = Counter(s["visible_handles"].index(s["correct_operands"][0]) for p in pipelines if p["category"] == "multi_object" for s in p["steps"] if s["mechanism"] == "selection")
tot = sum(pos.values())
check("multi-object winner position uniform", max(pos.values()) / tot <= 1.6 / MO_N_OBJECTS, str({k: round(100*v/tot, 1) for k, v in sorted(pos.items())}))

for cat in ("property_conditioned", "multi_object"):
    cnt = Counter((p["gate_property"], p["margin_band"], p["steps"][0]["mechanism"], p["tier"]) for p in pipelines if p["category"] == cat)
    check(f"{cat}: gate x band x mechanism x tier cells equal", len(set(cnt.values())) == 1, f"{len(cnt)} cells x {set(cnt.values())} pipelines")
    for band in MARGIN_BANDS:
        ss = [s for p in pipelines if p["category"] == cat and p["margin_band"] == band for s in p["steps"] if s["decision_margin"] is not None]
        if not ss: continue
        inb = sum(_in_band(s["decision_margin"], s["gate_property"], band) for s in ss) / len(ss)
        check(f"{cat} {band}: steps in band", inb >= 0.90, f"{100*inb:.1f}% of {len(ss)} steps")

for t in TIER_ORDER:
    for cat in PER_CELL:
        n = sum(1 for p in pipelines if p["category"] == cat and p["tier"] == t)
        check(f"{cat}: {t} pipelines = 1/3", n == PER_CELL[cat] * len(DEPTHS) // 3, str(n))

# FACTORIAL BALANCE: the three shared mechanisms must have the same step share in both categories
for mech in ("threshold", "boolean", "derived"):
    n_pc = sum(1 for p in pipelines if p["category"] == "property_conditioned" for s in p["steps"] if s["mechanism"] == mech and p["gate_locality"] == "global")
    n_mo = sum(1 for p in pipelines if p["category"] == "multi_object" for s in p["steps"] if s["mechanism"] == mech and p["gate_locality"] == "global")
    check(f"factorial: {mech} steps equal across categories", n_pc == n_mo, f"PC {n_pc} / MO {n_mo}")
for bnd in ("tight", "wide"):
    n_pc = sum(1 for p in pipelines if p["category"] == "property_conditioned" and p["margin_band"] == bnd and p["gate_locality"] == "global" for s in p["steps"] if s["mechanism"] == "threshold")
    n_mo = sum(1 for p in pipelines if p["category"] == "multi_object" and p["margin_band"] == bnd and p["gate_locality"] == "global" for s in p["steps"] if s["mechanism"] == "threshold")
    check(f"factorial: threshold {bnd}-band steps equal across categories", n_pc == n_mo, f"PC {n_pc} / MO {n_mo}")
noop = sum(1 for p in pipelines for s in p["steps"] if s["is_noop"])
print(f"\n  [INFO] no-op steps {noop:,}/{n_steps:,} ({100*noop/n_steps:.1f}%)")
print("\n" + ("ALL CHECKS PASSED" if not fails else f"{len(fails)} FAILED: {fails}"))
assert not fails

## [CELL 9] Worked example — one gated step under all four conditions

In [ ]:
ex = next(p for p in pipelines if p["category"] == "property_conditioned" and p["margin_band"] == "wide" and p["depth"] == 2)
store = rebuild(ex); s0 = ex["steps"][0]
print("PIPELINE", ex["pipeline_id"], "| gate", ex["gate_property"], "| margin", s0["decision_margin"], "| band", s0["margin_band"])
print("true value:", GATE_VALUE[ex["gate_property"]](properties(store[s0["visible_handles"][0]])))
print("answer    :", s0["correct_tool"], s0["correct_operands"], s0["correct_args"])
for c in CONDITIONS:
    vis = [(h, store[h]) for h in s0["visible_handles"]]
    print("\n" + "=" * 74 + f"\n### {c}\n" + "=" * 74)
    print("Observation:\n" + render_state(vis, c)[:600])
    print("\nTask:\n" + render_instruction(s0["instruction"], s0["visible_handles"], c))

## [CELL 10] Write and download

In [ ]:
payload = {
    "experiment": "geometry_exp2", "version": 6, "self_contained": True, "random_seed": SEED,
    "conditions": CONDITIONS, "modes": MODES,
    "text_conditions": sorted(TEXT_CONDITIONS), "handle_conditions": sorted(HANDLE_CONDITIONS),
    "read_tools_in_primary_conditions": False,
    "read_tools": {"raw": [], "augmented": [], "handle": ["get_object"], "handle_sum": ["get_object"]},
    "read_tool_note": "get_object(id) returns the WKT of a handle and never a property; it does not consume the step; budget = n_visible per step",
    "exp4_describe_tool": {"name": DESCRIBE_TOOL, "conditions": ["raw", "handle"], "budget_rule": "n_visible_objects + 1"},
    "pc_derived_templates": PC_DERIVED, "arg_tol_derived": ARG_TOL_DERIVED,
    "exp1_p": EXP1_P, "sigma": {k: round(v, 5) for k, v in SIGMA.items()},
    "assumptions": {"copy_fidelity": ASSUME_COPY_FIDELITY, "read_stated": ASSUME_READ_STATED, "rank_stated": ASSUME_RANK_STATED},
    "margin_bands": MARGIN_BANDS, "vertex_count_bands": VC_BANDS, "multi_object_n_objects": MO_N_OBJECTS,
    "depths": DEPTHS, "per_cell": PER_CELL,
    "gates": {"multi_object": MO_GATES}, "composite_gates": sorted(COMPOSITE_GATES),
    "design": {"factorial": "{1 object, 4 objects} x {threshold, boolean, derived, selection} minus the empty cell (1 object x selection)",
               "shared_mechanisms": ["threshold", "boolean", "derived"], "shared_gates": ["area", "perimeter", "aspect_ratio", "edge_length_variance", "convex"],
               "multi_object_only": ["selection"], "local_controls": ["width", "vertex_count"],
               "note": "Category means over the shared mechanisms are directly comparable. Selection is multi-object's own mechanism and is reported beside them with its 0.25 floor."},
    "gate_locality": GATE_LOCALITY,
    "spec_operations": SPEC_OPS,
    "predicted_table4": rows,
    "corpus": [{"object_id": c["object_id"], "tier": c["tier"], "shape_type": c["shape_type"], "wkt": to_wkt(c["poly"]),
                "properties": properties(c["poly"]), "knobs": c["knobs"]} for c in corpus],
    "pipelines": pipelines,
}
json.dump(payload, open(OUT_PIPELINES, "w"), indent=1)
summary = {"version": 6, "n_corpus": len(corpus), "n_pipelines": len(pipelines), "n_steps": n_steps,
           "api_calls_per_model_upper_bound": n_steps * 8, "predicted_table4": rows,
           "by_category": {c: {"pipelines": sum(1 for p in pipelines if p["category"] == c),
                               "steps": sum(len(p["steps"]) for p in pipelines if p["category"] == c)} for c in PER_CELL},
           "by_gate_band": dict(Counter(f"{p['gate_property']}/{p['margin_band']}" for p in pipelines)),
           "by_tier": dict(Counter(p["tier"] for p in pipelines)), "by_depth": dict(Counter(p["depth"] for p in pipelines)),
           "operation_usage": dict(Counter(s["correct_tool"] for p in pipelines for s in p["steps"])), "noop_steps": noop}
json.dump(summary, open(OUT_SUMMARY, "w"), indent=1)
print(f"wrote {OUT_PIPELINES} ({OUT_PIPELINES.stat().st_size/1e6:.1f} MB) and {OUT_SUMMARY}")
try:
    from google.colab import files
    files.download(str(OUT_PIPELINES)); files.download(str(OUT_SUMMARY)); files.download("exp2_runtime.py")
except Exception:
    print("not in Colab; files are on disk")